In [1]:
import pandas as pd
from pathlib import Path

In [2]:
INPUT_FILE = "merged_data.csv"
OUTPUT_DIR = Path("processed")

In [3]:
OUTPUT_DIR.mkdir(exist_ok=True)


In [4]:
df = pd.read_csv(INPUT_FILE)

In [5]:
print("Raw shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Raw shape: (3755, 13)

Columns:
['t', 'cmdty', 'market_id', 'market_name', 'state_id', 'state_name', 'district_id', 'district_name', 'variety', 'p_min', 'p_max', 'p_modal', 'source_file']


In [6]:
df["t"] = pd.to_datetime(df["t"], errors="coerce")

In [7]:
price_columns = ["p_min", "p_max", "p_modal"]

In [8]:
for col in price_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
df = df.dropna(subset=["t"])

In [11]:
df = df[
    (df["cmdty"].str.lower() == "tomato") &
    (df["district_name"].str.lower() == "gautam buddha nagar")
].copy()

In [12]:
print("\nAfter filtering:", df.shape)


After filtering: (3755, 13)


In [13]:
print("\nDate range:")
print("Start:", df["t"].min())
print("End:", df["t"].max())


Date range:
Start: 2020-01-01 00:00:00
End: 2025-10-29 00:00:00


In [14]:
print("\nMarkets:")
print(df["market_id"].value_counts())


Markets:
market_id
1903    2461
2541    1205
2542      89
Name: count, dtype: int64


In [15]:
print("\nVarieties:")
print(df["variety"].value_counts())


Varieties:
variety
Deshi     2779
Hybrid     889
Other       87
Name: count, dtype: int64


In [16]:
print("\nExact duplicate rows:", df.duplicated().sum())


Exact duplicate rows: 0


In [17]:
clean_raw_path = OUTPUT_DIR / "tomato_clean.csv"

In [18]:
df.to_csv(clean_raw_path, index=False)
print(f"\nSaved cleaned raw data → {clean_raw_path}")


Saved cleaned raw data → processed\tomato_clean.csv


In [19]:
daily_counts = (
    df.groupby("t")
      .size()
      .sort_index()
)

print("\nDaily observation statistics:")
print(daily_counts.describe())

print("\nObservation count distribution:")
print(daily_counts.value_counts().sort_index())

print("\nSample dates with multiple observations:")
print(daily_counts[daily_counts > 1].head(20))


Daily observation statistics:
count    1858.000000
mean        2.020990
std         0.768308
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         4.000000
dtype: float64

Observation count distribution:
1    524
2    776
3    553
4      5
Name: count, dtype: int64

Sample dates with multiple observations:
t
2020-01-02    2
2020-01-03    2
2020-01-04    2
2020-01-05    2
2020-01-06    2
2020-01-07    2
2020-01-09    2
2020-01-10    3
2020-01-11    2
2020-01-12    3
2020-01-13    3
2020-01-14    3
2020-01-15    2
2020-01-16    3
2020-01-17    2
2020-01-18    3
2020-01-19    3
2020-01-20    3
2020-01-21    3
2020-01-22    2
dtype: int64


In [20]:
print("\nModal price by market:")
print(
    df.groupby("market_id")["p_modal"]
      .describe()
)

print("\nModal price by variety:")
print(
    df.groupby("variety")["p_modal"]
      .describe()
)


Modal price by market:
            count         mean          std    min     25%     50%     75%  \
market_id                                                                    
1903       2461.0  1823.555465  1064.929535  500.0  1110.0  1500.0  2270.0   
2541       1205.0  1773.307884   808.698814  500.0  1200.0  1600.0  2235.0   
2542         89.0   975.280899   163.262824  600.0   900.0   900.0  1100.0   

              max  
market_id          
1903       7500.0  
2541       4300.0  
2542       1400.0  

Modal price by variety:
          count         mean          std    min     25%     50%     75%  \
variety                                                                    
Deshi    2779.0  1738.426053   933.189230  500.0  1110.0  1500.0  2150.0   
Hybrid    889.0  2019.876265  1116.927739  540.0  1280.0  1770.0  2570.0   
Other      87.0   972.988506   164.034277  600.0   900.0   900.0  1100.0   

            max  
variety          
Deshi    7400.0  
Hybrid   7500.0  
Other  

In [22]:
model_df = df[
    (df["market_id"].isin([1903, 2541])) &
    (df["variety"].isin(["Deshi", "Hybrid"]))
].copy()

print("\nAfter market/variety filtering:", model_df.shape)

print("\nRemaining markets:")
print(model_df["market_id"].value_counts())

print("\nRemaining varieties:")
print(model_df["variety"].value_counts())

daily_df = (
    model_df
    .groupby("t", as_index=False)["p_modal"]
    .median()
    .rename(columns={
        "t": "date",
        "p_modal": "price"
    })
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nDaily series:")
print(daily_df.head())

print("\nDaily series shape:", daily_df.shape)

print("\nDaily date range:")
print("Start:", daily_df["date"].min())
print("End:", daily_df["date"].max())

daily_output = OUTPUT_DIR / "tomato_daily.csv"
daily_df.to_csv(daily_output, index=False)
print(f"\nSaved daily time series → {daily_output}")


After market/variety filtering: (3666, 13)

Remaining markets:
market_id
1903    2461
2541    1205
Name: count, dtype: int64

Remaining varieties:
variety
Deshi     2779
Hybrid     887
Name: count, dtype: int64

Daily series:
        date   price
0 2020-01-01  1150.0
1 2020-01-02  1100.0
2 2020-01-03  1125.0
3 2020-01-04  1200.0
4 2020-01-05  1350.0

Daily series shape: (1849, 2)

Daily date range:
Start: 2020-01-01 00:00:00
End: 2025-10-29 00:00:00

Saved daily time series → processed\tomato_daily.csv
